# FABRIC Resource Finder


## 1. Setup and Configuration

In [ ]:
import sys
from pathlib import Path

# Add parent directory to Python path to import modules
repo_root = Path.cwd().parent

print(f"✅ Python path configured")
print(f"   Repository root: {repo_root}")

### Install fabric-generic-cluster

Install from 
- pypi (https://pypi.org/project/fabric-generic-cluster) or
- git repo (https://github.com/mcevik0/fabric-generic-cluster)

In [ ]:
def install_from_pypi():
    print("Installing from PyPI...")
    !pip install --upgrade fabric-generic-cluster --quiet

In [ ]:
def install_from_local():
    print("Installing from local git clone...")
    !rm -rf fabric-generic-cluster
    !git clone https://github.com/mcevik0/fabric-generic-cluster.git
    %cd fabric-generic-cluster
    !pip install -e . --quiet
    %cd ..

#### Select method and install

Install from pypi - recommended.

In [ ]:
install_method = "pypi"   # or "local"

if install_method == "pypi":
    install_from_pypi()
elif install_method == "local":
    install_from_local()
else:
    print("Unknown installation method:", install_method)

### Configure

In [ ]:
# Import modules

from fabric_generic_cluster import load_topology_from_yaml_file, SiteTopology
from fabric_generic_cluster import deployment as sd
from fabric_generic_cluster import network_config as snc
from fabric_generic_cluster import ssh_setup as ssh
from fabric_generic_cluster import ansible_setup as ansible
from fabric_generic_cluster import selinux_management as selinux

from fabric_generic_cluster import (
    find_sites_with_resources,
    find_hosts_with_resources,
    validate_topology,
    find_hosts_for_topology,
)

print("✅ Modules imported successfully")

In [ ]:
# Show Fablib configuration

sd.show_config()

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()
    print("✅ Fablib initialized successfully")
    fablib.show_config()
except Exception as e:
    print(f"❌ Exception: {e}")
    

## 2. Helper Function - Get Sites DataFrame

In [ ]:
import pandas as pd

def get_sites_dataframe(fablib, force_refresh=False):
    """
    Get all FABRIC sites with their resources as a DataFrame.
    
    Args:
        fablib: FablibManager instance
        
    Returns:
        pandas.DataFrame with site information and resources
    """
    resources = fablib.get_resources(force_refresh=force_refresh)
    sites_list = []

    for site_name in resources.sites:
        site = resources.sites[site_name]
        
        # Build site dictionary
        site_dict = {'name': site_name}
        
        # Basic attributes
        try:
            site_dict['state'] = site.get_state()
        except:
            site_dict['state'] = None
        
        # Get location info
        try:
            location = site.get_location_postal()
            site_dict['address'] = location
        except:
            site_dict['address'] = None
        
        # Get basic resources
        try:
            site_dict['cores_available'] = site.get_core_available()
            site_dict['cores_capacity'] = site.get_core_capacity()
            site_dict['cores_allocated'] = site.get_core_allocated()
            
            site_dict['ram_available'] = site.get_ram_available()
            site_dict['ram_capacity'] = site.get_ram_capacity()
            site_dict['ram_allocated'] = site.get_ram_allocated()
            
            site_dict['disk_available'] = site.get_disk_available()
            site_dict['disk_capacity'] = site.get_disk_capacity()
            site_dict['disk_allocated'] = site.get_disk_allocated()
        except:
            pass
        
        # Get component info from site_info (NICs, GPUs, FPGAs, NVMe, etc.)
        try:
            site_info = site.site_info
            for component_name, component_data in site_info.items():
                if isinstance(component_data, dict) and 'capacity' in component_data:
                    # Create columns for each component
                    site_dict[f'{component_name}_capacity'] = component_data.get('capacity', 0)
                    site_dict[f'{component_name}_allocated'] = component_data.get('allocated', 0)
                    site_dict[f'{component_name}_available'] = (
                        component_data.get('capacity', 0) - component_data.get('allocated', 0)
                    )
        except Exception as e:
            pass
        
        sites_list.append(site_dict)

    return pd.DataFrame(sites_list)


## 3. Query - Sites with SmartNIC ConnectX-5

In [ ]:
print("🔎 Sites with SmartNIC ConnectX-5 available:\n")

sites_df = get_sites_dataframe(fablib)
cx5_sites = sites_df[sites_df['smartnic-connectx-5_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-5_available']
print(cx5_sites[display_cols].to_string(index=False))


## 4. Query - Sites with SmartNIC ConnectX-6

In [ ]:
print("🔎 Sites with SmartNIC ConnectX-6 available:\n")

sites_df = get_sites_dataframe(fablib)
cx6_sites = sites_df[sites_df['smartnic-connectx-6_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-6_available']
print(cx6_sites[display_cols].to_string(index=False))


## 5. Query - Sites with SharedNIC ConnectX-6

In [ ]:
print("🔎 Sites with SharedNIC ConnectX-6 available:\n")

sites_df = get_sites_dataframe(fablib)
shared_cx6_sites = sites_df[sites_df['sharednic-connectx-6_available'].fillna(0) > 0]

display_cols = ['name', 'sharednic-connectx-6_available']
print(shared_cx6_sites[display_cols].to_string(index=False))


## 6. Query - Sites with DPU NICs (ConnectX-7 100G, ConnectX-7 400G)

In [ ]:
print("🔎 Sites with DPU NICs (ConnectX-7 100G) available:\n")

sites_df = get_sites_dataframe(fablib)
dpu_sites = sites_df[sites_df['smartnic-connectx-7-100_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-7-100_available']
print(dpu_sites[display_cols].to_string(index=False))


In [ ]:
print("🔎 Sites with DPU NICs (ConnectX-7 400G) available:\n")

sites_df = get_sites_dataframe(fablib)
dpu_sites = sites_df[sites_df['smartnic-connectx-7-400_available'].fillna(0) > 0]

display_cols = ['name', 'smartnic-connectx-7-400_available']
print(dpu_sites[display_cols].to_string(index=False))


## 7. Query - Sites with BOTH SmartNIC ConnectX-5 and ConnectX-6

In [ ]:
print("🔎 Sites with both SmartNIC ConnectX-5 and ConnectX-6:\n")

sites_df = get_sites_dataframe(fablib)
both_nics_sites = sites_df[
    (sites_df['smartnic-connectx-5_available'].fillna(0) > 0) & 
    (sites_df['smartnic-connectx-6_available'].fillna(0) > 0)
]

display_cols = ['name', 'smartnic-connectx-5_available', 'smartnic-connectx-6_available']
print(both_nics_sites[display_cols].to_string(index=False))


## 8. Query - Sites with GPUs

In [ ]:
print("🔎 Sites with RTX6000 GPUs:\n")

sites_df = get_sites_dataframe(fablib)
rtx6000_sites = sites_df[sites_df['gpu-rtx6000_available'].fillna(0) > 0]

display_cols = ['name', 'gpu-rtx6000_available']
print(rtx6000_sites[display_cols].to_string(index=False))

print("\n🔎 Sites with Tesla T4 GPUs:\n")

t4_sites = sites_df[sites_df['gpu-tesla t4_available'].fillna(0) > 0]
display_cols = ['name', 'gpu-tesla t4_available']
print(t4_sites[display_cols].to_string(index=False))

print("\n🔎 Sites with A30 GPUs:\n")

a30_sites = sites_df[sites_df['gpu-a30_available'].fillna(0) > 0]
display_cols = ['name', 'gpu-a30_available']
print(a30_sites[display_cols].to_string(index=False))


## 9. Query - Sites with FPGAs

In [ ]:
print("🔎 Sites with Xilinx U280 FPGAs:\n")

sites_df = get_sites_dataframe(fablib)
fpga_sites = sites_df[sites_df['fpga-xilinx-u280_available'].fillna(0) > 0]

display_cols = ['name', 'fpga-xilinx-u280_available']
print(fpga_sites[display_cols].to_string(index=False))


## 10. Query - Sites with NVMe Storage

In [ ]:
print("🔎 Sites with NVMe storage:\n")

sites_df = get_sites_dataframe(fablib)
nvme_sites = sites_df[sites_df['nvme-p4510_available'].fillna(0) > 0]

display_cols = ['name', 'nvme-p4510_available']
print(nvme_sites[display_cols].to_string(index=False))


## 11. Custom Resource Finder Function

### Resources of the Sites

#### Example: Find sites with at least 256 cores, 300 GB RAM, and SharedNIC ConnectX-6

In [ ]:
find_sites_with_resources(
    fablib,
    min_cores=256,
    min_ram=300,
    min_disk=500,
    sharednic_connectx_6=1,
    force_refresh=True,
    return_data=False
)

In [ ]:
find_sites_with_resources(
    fablib, min_cores=100, 
    min_ram=200, 
    gpu_a30=1, 
    force_refresh=True, 
    return_data=False
)

### Resources of the Hosts (Workers)

#### Example: Find hosts with at least 8 cores, 32GB RAM, and an A30 GPU

In [ ]:
find_hosts_with_resources(
    fablib,
    min_cores=8,
    min_ram=32,
    gpu_a30=1,
    force_refresh=True,
    return_data=False
)

In [ ]:
find_hosts_with_resources(
    fablib,
    min_cores=20,
    min_ram=64,
    gpu_rtx6000=1,
    force_refresh=True,
    return_data=False
)

## 12. Find Best Site for Your Topology

### Load Topology from the Model

<div class="alert alert-block alert-info">
Define the <b>YAML Directory</b> and <b>Model File</b>
</div>

In [ ]:
YAML_DIR = repo_root / "model"
site_topology_yaml = "../model/m10.yml"

print(f"✅ YAML directory: {YAML_DIR}")
print(f"✅ YAML file: {site_topology_yaml}")

In [ ]:
# Load and validate topology (raises ValidationError if invalid)
try:
    topology = load_topology_from_yaml_file(site_topology_yaml)
    print("✅ Topology loaded and validated successfully!")
    print(f"   Nodes: {len(topology.site_topology_nodes.nodes)}")
    print(f"   Networks: {len(topology.site_topology_networks.networks)}")

    result = validate_topology(
        fablib,
        topology,
        sites_prefer=[],  # Optional: prefer certain sites ['WASH', 'SRI']
        return_data=True,
    )
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise


# Access results
if result and result['can_deploy']:
    print("\n✅ Topology validated — run find_hosts_for_topology() to see candidate hosts.")

#### Example: Validate topology against FABRIC resources

In [ ]:
validate_topology(
    fablib,
    topology,
    sites_prefer=['WASH', 'SRI'],
)

#### Example: Find candidate hosts per node

In [ ]:
find_hosts_for_topology(
    fablib,
    topology,
    sites_prefer=['WASH', 'SRI'],
    sites_avoid=[],
)

## 13. Export Site Availability to File

In [ ]:
def export_site_availability(fablib, filename="site_availability.txt"):
    """
    Export current site availability to file.
    
    Args:
        fablib: FablibManager instance
        filename: Output filename
    """
    from datetime import datetime
    
    # Get all sites
    sites_df = get_sites_dataframe(fablib)
    
    # Add timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    with open(filename, 'w') as f:
        f.write(f"FABRIC Site Availability Report\n")
        f.write(f"Generated: {timestamp}\n")
        f.write("=" * 80 + "\n\n")
        f.write(sites_df.to_string())
    
    print(f"✅ Exported to: {filename}")

In [ ]:
# Uncomment to export
export_site_availability(fablib)

## 14. Get All Available Resource Types

In [ ]:
# Get all sites as DataFrame
sites_df = get_sites_dataframe(fablib)

# Find columns related to hardware
hardware_columns = [
    col for col in sites_df.columns 
    if any(hw in col.lower() for hw in [
        'nic', 'gpu', 'fpga', 'nvme', 'cores', 'ram', 'disk', 
        'connectx', 'tesla', 'rtx', 'a30', 'a40',
        'u280', 'smartnic', 'sharednic'
    ]) and 'available' in col.lower()
]

print("📋 Available Resource Types in FABRIC:\n")
for col in sorted(hardware_columns):
    if col in sites_df.columns:
        # Handle NaN values properly
        total_available = sites_df[col].fillna(0).sum() if pd.api.types.is_numeric_dtype(sites_df[col]) else "N/A"
        if isinstance(total_available, float):
            total_available = int(total_available)
        print(f"   • {col}: {total_available}")